In [43]:
import pandas as pd

df = pd.read_csv("diabetes.csv")
df.head(), df.count()

(     id   chol  stab.glu   hdl  ratio  glyhb    location  age  gender  height  \
 0  1000  203.0        82  56.0    3.6   4.31  Buckingham   46  female    62.0   
 1  1001  165.0        97  24.0    6.9   4.44  Buckingham   29  female    64.0   
 2  1002  228.0        92  37.0    6.2   4.64  Buckingham   58  female    61.0   
 3  1003   78.0        93  12.0    6.5   4.63  Buckingham   67    male    67.0   
 4  1005  249.0        90  28.0    8.9   7.72  Buckingham   64    male    68.0   
 
    weight   frame  bp.1s  bp.1d  bp.2s  bp.2d  waist   hip  time.ppn  
 0   121.0  medium  118.0   59.0    NaN    NaN   29.0  38.0     720.0  
 1   218.0   large  112.0   68.0    NaN    NaN   46.0  48.0     360.0  
 2   256.0   large  190.0   92.0  185.0   92.0   49.0  57.0     180.0  
 3   119.0   large  110.0   50.0    NaN    NaN   33.0  38.0     480.0  
 4   183.0  medium  138.0   80.0    NaN    NaN   44.0  41.0     300.0  ,
 id          403
 chol        402
 stab.glu    403
 hdl         402
 rati

In [44]:
df.isnull().sum()

id            0
chol          1
stab.glu      0
hdl           1
ratio         1
glyhb        13
location      0
age           0
gender        0
height        5
weight        1
frame        12
bp.1s         5
bp.1d         5
bp.2s       262
bp.2d       262
waist         2
hip           2
time.ppn      3
dtype: int64

In [45]:
# since the bp.2d , bp.2s the missing is more than the half so I need to drop them so dont get overfitting
# drop the id because it is not a medical feature
# drop the location because this don't have medical value
df = df.drop(columns=['bp.2s', 'bp.2d', 'id', 'location'])

# since the glyhb is what we want to predict so I will drop the missing values only
df = df.dropna(subset=['glyhb'])

numerical = [
    'chol', 'hdl', 'ratio', 'age', 'height', 'weight',
    'bp.1s', 'bp.1d', 'waist', 'hip', 'time.ppn'
]
categorical = ['gender', 'frame']



In [46]:
# the target is binary , 1 if the patient is diabetic and 0 if not
# there is no Type 1 in the data (nobody under 25) so I dont split to Type 1 and Type 2
df['target'] = ((df['glyhb'] >= 6.5) | (df['stab.glu'] >= 126)).astype(int)
df['target'].value_counts()



target
0    310
1     80
Name: count, dtype: int64

In [47]:
# now I want Risk Ratio , Risk Difference
global_risk = df['target'].mean()
global_risk

for col in categorical:
    print(f"\nFeature: {col}")
    df_group = df.groupby(col, observed=False)['target'].agg(
        Count='count',
        Risk_Rate='mean'
    )

    df_group['Risk_Ratio'] = df_group['Risk_Rate'] / global_risk
    df_group['Risk_Difference'] = df_group['Risk_Rate'] - global_risk

    print(df_group.to_string())

for col in numerical:
    print(f"\nFeature: {col}")
    col_binned = pd.qcut(df[col], q=4, duplicates='drop')

    df_group = df.groupby(col_binned, observed=False)['target'].agg(
        Count='count',
        Risk_Rate='mean'
    )

    df_group['Risk_Ratio'] = df_group['Risk_Rate'] / global_risk
    df_group['Risk_Difference'] = df_group['Risk_Rate'] - global_risk

    print(df_group.to_string())




Feature: gender
        Count  Risk_Rate  Risk_Ratio  Risk_Difference
gender                                               
female    228   0.188596    0.919408        -0.016532
male      162   0.228395    1.113426         0.023267

Feature: frame
        Count  Risk_Rate  Risk_Ratio  Risk_Difference
frame                                                
large      99   0.333333    1.625000         0.128205
medium    178   0.196629    0.958567        -0.008499
small     102   0.098039    0.477941        -0.107089

Feature: chol
                 Count  Risk_Rate  Risk_Ratio  Risk_Difference
chol                                                          
(77.999, 179.0]    106   0.122642    0.597877        -0.082487
(179.0, 203.0]      89   0.179775    0.876404        -0.025353
(203.0, 229.0]      97   0.226804    1.105670         0.021676
(229.0, 443.0]      97   0.298969    1.457474         0.093841

Feature: hdl
                Count  Risk_Rate  Risk_Ratio  Risk_Difference
hdl         

**from the Risk Ratio , Risk Difference the data is not give me overfitting from train on feature that don't give the model information or value**

In [48]:

from sklearn.model_selection import train_test_split

# stratify so the diabetic ratio is the same in train , val and test
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1, stratify=df['target'])
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1, stratify=df_full_train['target'])

y_train = df_train['target'].values
y_val = df_val['target'].values
y_test = df_test['target'].values

# glyhb and stab.glu are dropped from the features so dont get data leakage
# since they are the direct diagnostic indicators that created the target
leak_cols = ['target', 'glyhb', 'stab.glu']
df_train = df_train.drop(columns=leak_cols)
df_val = df_val.drop(columns=leak_cols)
df_test = df_test.drop(columns=leak_cols)

#----------------------------


In [49]:

# imputing the val and test sets using the train statistics so the model remains unbiased
# in the numerical columns I use the median because I need medical model
for col in numerical:
    median = df_train[col].median()
    df_train[col] = df_train[col].fillna(median)
    df_val[col] = df_val[col].fillna(median)
    df_test[col] = df_test[col].fillna(median)

# for the categorical columns I will use the most used value (the mode of the train)
for col in categorical:
    mode = df_train[col].mode()[0]
    df_train[col] = df_train[col].fillna(mode)
    df_val[col] = df_val[col].fillna(mode)
    df_test[col] = df_test[col].fillna(mode)

df_train.isnull().sum()



chol        0
hdl         0
ratio       0
age         0
gender      0
height      0
weight      0
frame       0
bp.1s       0
bp.1d       0
waist       0
hip         0
time.ppn    0
dtype: int64

In [ ]:
# the BMI is feature that very important to discover T2D (Type 2 Diabetes)
# BMI = (weight * 703) / (height**2)
# waist to hip ratio is also a known risk factor for diabetes
import numpy as np

for dataset in [df_train, df_val, df_test]:
    dataset['BMI'] = (dataset['weight'] * 703) / (dataset['height'] ** 2)
    dataset['waist_hip'] = dataset['waist'] / dataset['hip']
    # **During training, I discovered that the model's accuracy did not exceed 81%, so I plan to try increasing the number of features using a medical dataset I found. so I will try anouther feature than BMI like whtr , pulse_pressure , map**
    # 2. Waist-to-Height Ratio (Superior predictor for visceral adiposity & insulin resistance)
    dataset['whtr'] = dataset['waist'] / (dataset['height'] + 1e-5)
    
    # # 3. Pulse Pressure (Proxy for arterial stiffness and vascular compliance)
    # dataset['pulse_pressure'] = dataset['bp.1s'] - dataset['bp.1d']
    
    # # 4. Mean Arterial Pressure (MAP)
    # dataset['map'] = dataset['bp.1d'] + (dataset['pulse_pressure'] / 3.0)
    
    # # 5. Non-linear & Interaction Terms (To capture metabolic syndrome co-factors)
    # dataset['age_squared'] = dataset['age'] ** 2
    # dataset['bmi_age'] = dataset['BMI'] * dataset['age']
    # dataset['bp_age_interaction'] = dataset['bp.1s'] * dataset['age']
    # dataset['chol_bmi_interaction'] = dataset['chol'] * dataset['BMI']
    
    # # 6. Logarithmic transformations for skewed biochemical variables
    # dataset['log_chol'] = np.log1p(dataset['chol'])
    # dataset['log_hdl'] = np.log1p(dataset['hdl'])

In [67]:
from sklearn.feature_extraction import DictVectorizer

dv = DictVectorizer(sparse=False)

X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))
X_test = dv.transform(df_test.to_dict(orient='records'))


In [68]:

from sklearn.metrics import make_scorer, fbeta_score
from sklearn.model_selection import StratifiedKFold

# f2 give the recall more weight than the precision
# because I dont want to tell the sick patient that he is healthy
scoring = {
    'recall': 'recall',
    'f1': 'f1',
    'accuracy': 'accuracy',
    'f2': make_scorer(fbeta_score, beta=2),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
df_train

,chol,hdl,ratio,age,gender,height,weight,frame,bp.1s,bp.1d,waist,hip,time.ppn,BMI,waist_hip,whtr
291,162.0,40.0,4.1,43,male,67.0,216.0,large,100.0,70.0,41.0,44.0,30.0,33.826687,0.931818,0.611940
48,169.0,51.0,3.3,40,female,65.0,180.0,medium,106.0,82.0,40.0,44.0,780.0,29.950296,0.909091,0.615385
285,277.0,62.0,4.5,60,female,61.0,128.0,small,140.0,86.0,33.0,39.0,240.0,24.182747,0.846154,0.540984
310,179.0,60.0,3.0,26,female,60.0,130.0,small,138.0,84.0,32.0,40.0,270.0,25.386111,0.800000,0.533333
169,212.0,49.0,4.3,51,female,65.0,145.0,small,230.0,120.0,38.0,42.0,60.0,24.126627,0.904762,0.584615
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
352,226.0,65.0,3.5,27,male,69.0,289.0,large,130.0,100.0,48.0,51.0,75.0,42.673178,0.941176,0.695652
252,241.0,40.0,6.0,27,female,63.0,179.0,medium,120.0,75.0,40.0,42.0,720.0,31.704963,0.952381,0.634921
148,269.0,34.0,7.9,41,female,62.0,160.0,medium,126.0,90.0,39.0,41.0,390.0,29.261186,0.951220,0.629032
135,209.0,34.0,6.1,48,female,63.0,121.0,small,111.0,62.0,32.0,38.0,855.0,21.431847,0.842105,0.507936


In [69]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# healthy / diabetic ratio , used in xgboost because it dont have class_weight
neg_pos_ratio = (y_train == 0).sum() / (y_train == 1).sum()

# the logistic regression need scaling because chol and weight are big numbers
models = {
    'Logistic Regression': (
        make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, class_weight='balanced', random_state=42)),
        {
            'logisticregression__C': [0.001, 0.01, 0.1, 1, 10, 100],
            'logisticregression__solver': ['lbfgs', 'newton-cg'],
        }
    ),
    'Decision Tree': (
        DecisionTreeClassifier(class_weight='balanced', random_state=42),
        {
            'max_depth': [2, 3, 4, 5, 6, 8, None],
            'min_samples_split': [2, 5, 10, 20],
            'min_samples_leaf': [1, 2, 5, 10],
            'criterion': ['gini', 'entropy'],
        }
    ),
    'Random Forest': (
        RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
        {
            'n_estimators': [100, 200, 400],
            'max_depth':  [2, 3, 4, 5, 6, 8, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2'],
        }
    ),
    'XGBoost': (
        XGBClassifier(eval_metric='logloss', scale_pos_weight=neg_pos_ratio, random_state=42, n_jobs=-1),
        {
            'n_estimators': [100, 200, 400],
            'max_depth':  [2, 3, 4, 5, 6, 8],
            'learning_rate': [0.01, 0.05, 0.1, 0.2,0.3,0.08],
            'subsample': [0.7, 1.0],
            'colsample_bytree': [0.7, 1.0],
            'min_child_weight': [1, 3, 5],
        }
    ),
}


In [70]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import recall_score, f1_score, accuracy_score, confusion_matrix

best_models = {}
val_results = []

for name, (model, params) in models.items():
    grid = GridSearchCV(model, params, scoring=scoring, refit='f2', cv=cv, n_jobs=-1)
    grid.fit(X_train, y_train)

    best_models[name] = grid.best_estimator_
    i = grid.best_index_
    print(f"\n{name}")
    print("Best params:", grid.best_params_)
    print(f"CV recall: {grid.cv_results_['mean_test_recall'][i]:.4f}  CV f1: {grid.cv_results_['mean_test_f1'][i]:.4f}  CV accuracy: {grid.cv_results_['mean_test_accuracy'][i]:.4f}")

    pred = grid.best_estimator_.predict(X_val)
    val_results.append({
        'model': name,
        'recall': recall_score(y_val, pred),
        'f1': f1_score(y_val, pred),
        'accuracy': accuracy_score(y_val, pred),
        'f2': fbeta_score(y_val, pred, beta=2),
    })
    print(confusion_matrix(y_val, pred))

val_df = pd.DataFrame(val_results).set_index('model')
val_df.round(4)



Logistic Regression
Best params: {'logisticregression__C': 0.1, 'logisticregression__solver': 'lbfgs'}
CV recall: 0.6244  CV f1: 0.4643  CV accuracy: 0.7095
[[49 13]
 [ 4 12]]

Decision Tree
Best params: {'criterion': 'entropy', 'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 10}
CV recall: 0.7244  CV f1: 0.4952  CV accuracy: 0.7005
[[44 18]
 [ 5 11]]

Random Forest
Best params: {'max_depth': 2, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
CV recall: 0.7067  CV f1: 0.5250  CV accuracy: 0.7394
[[45 17]
 [ 7  9]]

XGBoost
Best params: {'colsample_bytree': 1.0, 'learning_rate': 0.01, 'max_depth': 2, 'min_child_weight': 3, 'n_estimators': 100, 'subsample': 1.0}
CV recall: 0.7244  CV f1: 0.5381  CV accuracy: 0.7476
[[44 18]
 [ 6 10]]


,recall,f1,accuracy,f2
model,,,,
Logistic Regression,0.7500,0.5854,0.7821,0.6742
Decision Tree,0.6875,0.4889,0.7051,0.5914
Random Forest,0.5625,0.4286,0.6923,0.5000
XGBoost,0.6250,0.4545,0.6923,0.5435


In [72]:
from sklearn.metrics import classification_report

# test set only once with the best model on validation
best_name = val_df['f1'].idxmax()
print(f"Best model on validation: {best_name}")

pred_test = best_models[best_name].predict(X_test)
print(classification_report(
    y_test,
    pred_test,
    target_names=['Non-Diabetic (0)', 'Diabetic (1)'],
    zero_division=0
))

cm = confusion_matrix(y_test, pred_test)
print(cm)
print(f"Diabetic patients predicted as healthy: {cm[1, 0]} out of {cm[1].sum()}")


Best model on validation: Logistic Regression
                  precision    recall  f1-score   support

Non-Diabetic (0)       0.89      0.66      0.76        62
    Diabetic (1)       0.34      0.69      0.46        16

        accuracy                           0.67        78
       macro avg       0.62      0.67      0.61        78
    weighted avg       0.78      0.67      0.70        78

[[41 21]
 [ 5 11]]
Diabetic patients predicted as healthy: 5 out of 16


In [63]:
# I am currently facing an issue with low model accuracy, which stems from a scarcity of actual "positive" (affected/diseased) samples. Since my model relies on these samples to predict attributes like age, weight, and similar variables, I need a solution to this problem.

In [64]:
import numpy as np
from sklearn.model_selection import learning_curve

# learning curve for the logistic regression to see if more data will help
sizes, train_scores, val_scores = learning_curve(
    best_models['Logistic Regression'], X_train, y_train,
    train_sizes=np.linspace(0.3, 1.0, 5), cv=cv, scoring=scoring['f2'], n_jobs=-1
)

for s, tr, va in zip(sizes, train_scores.mean(axis=1), val_scores.mean(axis=1)):
    print(f"size: {s}  train f2: {tr:.3f}  val f2: {va:.3f}")

size: 56  train f2: 0.760  val f2: 0.478
size: 88  train f2: 0.721  val f2: 0.507
size: 121  train f2: 0.697  val f2: 0.536
size: 154  train f2: 0.691  val f2: 0.570
size: 187  train f2: 0.639  val f2: 0.548


In [65]:
# from the learning curve this is not problem from the data is small 

In [59]:
from sklearn.model_selection import cross_val_predict

# probabilities from the cross validation on the train so the threshold is not chosen on only 16 patients
proba = cross_val_predict(best_models['Random Forest'], X_train, y_train, cv=cv, method='predict_proba')[:, 1]

for t in [0.2, 0.3, 0.4, 0.5]:
    pred = (proba >= t).astype(int)
    print(f"threshold: {t}  recall: {recall_score(y_train, pred):.3f}  f1: {f1_score(y_train, pred):.3f}  accuracy: {accuracy_score(y_train, pred):.3f}")

threshold: 0.2  recall: 0.979  f1: 0.349  accuracy: 0.252
threshold: 0.3  recall: 0.958  f1: 0.405  accuracy: 0.423
threshold: 0.4  recall: 0.812  f1: 0.433  accuracy: 0.564
threshold: 0.5  recall: 0.729  f1: 0.538  accuracy: 0.744
